# Import One Topic — Pipeline Notebook

Run each cell in order. Edit the **Configuration** cell below to set your topic, database, and parameters.

Each step can be re-run independently. Progress is checkpointed automatically.

Extract metadata from openalex

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
fetch_openalex_taxonomy.py — Download the full OpenAlex topic hierarchy with paper counts.

Fetches all 4 levels of the OpenAlex classification system and saves them as JSON and CSV:
  Level 1 — Domains   (~5)
  Level 2 — Fields    (~25)
  Level 3 — Subfields (~250)  ← these are the Mapo galaxy groupings
  Level 4 — Topics    (~4500)

Outputs (written to OUTPUT_DIR):
  domains.json / domains.csv
  fields.json  / fields.csv
  subfields.json / subfields.csv
  topics.json  / topics.csv
  taxonomy_summary.json   — full nested hierarchy with counts

Usage:
  python fetch_openalex_taxonomy.py
"""
import csv
import pandas as pd
import json
import os
import time
import urllib.parse
import urllib.request
from typing import Any, Dict, List, Optional

# ──────────────────────────────────────────────
# PARAMETERS — adjust as needed
# ──────────────────────────────────────────────

EMAIL = "tom.hirsch3000@gmail.com"   # OpenAlex polite-pool email
OUTPUT_DIR = "openalex_taxonomy"      # Directory to write output files
PER_PAGE = 200                        # Max results per page (OpenAlex max is 200)
THROTTLE_S = 0.12                     # Seconds between requests (polite pool: ~10 req/s)
MAX_RETRIES = 5                       # Retry attempts on transient errors

# Set to True to also save a flattened CSV with every topic's full ancestry
SAVE_FULL_FLAT_CSV = True

# Minimum works_count to include a topic in the galaxy candidate list
# (set to 0 to include everything)
MIN_WORKS_FOR_GALAXY = 0

# ──────────────────────────────────────────────

In [4]:
BASE_URL = "https://api.openalex.org"

ENDPOINTS = {
    "domains":   f"{BASE_URL}/domains",
    "fields":    f"{BASE_URL}/fields",
    "subfields": f"{BASE_URL}/subfields",
    "topics":    f"{BASE_URL}/topics",
}

# Fields to select per level — keeps responses small and focused
SELECT_FIELDS = {
    "domains":   "id,display_name,description,works_count,cited_by_count",
    "fields":    "id,display_name,description,works_count,cited_by_count,domain",
    "subfields": "id,display_name,description,works_count,cited_by_count,field,domain",
    "topics":    "id,display_name,description,works_count,cited_by_count,subfield,field,domain,keywords",
}


def fetch_page(url: str, params: Dict[str, str], attempt: int = 0) -> Optional[Dict]:
    full_url = url + "?" + urllib.parse.urlencode(params)
    try:
        req = urllib.request.Request(
            full_url,
            headers={"User-Agent": f"mapo-research/1.0 (mailto:{EMAIL})"},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        if attempt >= MAX_RETRIES:
            print(f"  [error] All {MAX_RETRIES} retries failed: {e}")
            return None
        wait = 2 ** attempt
        print(f"  [warn] Attempt {attempt+1} failed ({e}) — retrying in {wait}s")
        time.sleep(wait)
        return fetch_page(url, params, attempt + 1)


def fetch_all(level: str) -> List[Dict]:
    """Paginate through all results for a given level endpoint."""
    url = ENDPOINTS[level]
    select = SELECT_FIELDS[level]
    items: List[Dict] = []
    cursor = "*"
    page_num = 0

    print(f"\n[{level}] Fetching…")
    while True:
        params = {
            "select": select,
            "per-page": str(PER_PAGE),
            "cursor": cursor,
            "mailto": EMAIL,
        }
        data = fetch_page(url, params)
        if not data:
            print(f"  [error] Empty response on page {page_num + 1}")
            break

        results = data.get("results", [])
        meta = data.get("meta", {})
        items.extend(results)
        page_num += 1

        total = meta.get("count", "?")
        print(f"  Page {page_num}: +{len(results)} items  (total so far: {len(items)}/{total})")

        next_cursor = meta.get("next_cursor")
        if not next_cursor or not results:
            break
        cursor = next_cursor
        time.sleep(THROTTLE_S)

    print(f"  -> Done: {len(items)} {level} fetched")
    return items


def strip_id(oa_id: str) -> str:
    """'https://openalex.org/T12345' -> 'T12345'"""
    return oa_id.replace("https://openalex.org/", "") if oa_id else ""


def flatten_item(item: Dict, level: str) -> Dict:
    """Normalise a raw API item into a flat dict with consistent keys."""
    flat: Dict[str, Any] = {
        "id":           strip_id(item.get("id", "")),
        "display_name": item.get("display_name", ""),
        "description":  item.get("description", ""),
        "works_count":  item.get("works_count", 0),
        "cited_by_count": item.get("cited_by_count", 0),
    }

    if level in ("fields", "subfields", "topics"):
        domain = item.get("domain") or {}
        flat["domain_id"]   = strip_id(domain.get("id", ""))
        flat["domain_name"] = domain.get("display_name", "")

    if level in ("subfields", "topics"):
        field = item.get("field") or {}
        flat["field_id"]   = strip_id(field.get("id", ""))
        flat["field_name"] = field.get("display_name", "")

    if level == "topics":
        subfield = item.get("subfield") or {}
        flat["subfield_id"]   = strip_id(subfield.get("id", ""))
        flat["subfield_name"] = subfield.get("display_name", "")
        keywords = item.get("keywords") or []
        flat["keywords"] = "; ".join(keywords) if isinstance(keywords, list) else str(keywords)
        ids = item.get("ids") or {}
        flat["wikipedia"] = ids.get("wikipedia", "") or item.get("wikipedia", "")

    return flat


def save_json(data: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  Saved: {path}")


def save_csv(rows: List[Dict], path: str):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"  Saved: {path}")


def build_nested_hierarchy(
    domains: List[Dict],
    fields: List[Dict],
    subfields: List[Dict],
    topics: List[Dict],
) -> List[Dict]:
    """Build a fully nested dict: domain > field > subfield > [topics]."""
    # Index everything by ID for fast lookup
    fields_by_domain: Dict[str, List] = {}
    for f in fields:
        did = f["domain_id"]
        fields_by_domain.setdefault(did, []).append(f)

    subfields_by_field: Dict[str, List] = {}
    for sf in subfields:
        fid = sf["field_id"]
        subfields_by_field.setdefault(fid, []).append(sf)

    topics_by_subfield: Dict[str, List] = {}
    for t in topics:
        sfid = t["subfield_id"]
        topics_by_subfield.setdefault(sfid, []).append(t)

    hierarchy = []
    for d in sorted(domains, key=lambda x: x["display_name"]):
        domain_entry = {**d, "fields": []}
        for f in sorted(fields_by_domain.get(d["id"], []), key=lambda x: x["display_name"]):
            field_entry = {**f, "subfields": []}
            for sf in sorted(subfields_by_field.get(f["id"], []), key=lambda x: x["display_name"]):
                sf_entry = {
                    **sf,
                    "topics": sorted(
                        topics_by_subfield.get(sf["id"], []),
                        key=lambda x: -x["works_count"],
                    ),
                }
                field_entry["subfields"].append(sf_entry)
            domain_entry["fields"].append(field_entry)
        hierarchy.append(domain_entry)
    return hierarchy


def print_summary(domains, fields, subfields, topics):
    print("\n" + "=" * 60)
    print("TAXONOMY SUMMARY")
    print("=" * 60)
    print(f"  Domains:   {len(domains):>5}")
    print(f"  Fields:    {len(fields):>5}")
    print(f"  Subfields: {len(subfields):>5}  ← Mapo galaxy groups")
    print(f"  Topics:    {len(topics):>5}")

    total_works = sum(d["works_count"] for d in domains)
    print(f"\n  Total works indexed by OpenAlex: {total_works:,}")

    print("\nTop 10 subfields by paper count:")
    top_sf = sorted(subfields, key=lambda x: -x["works_count"])[:10]
    for i, sf in enumerate(top_sf, 1):
        print(f"  {i:>2}. {sf['display_name']:<40} {sf['works_count']:>10,}  ({sf['field_name']})")

    print("\nDomains:")
    for d in sorted(domains, key=lambda x: -x["works_count"]):
        print(f"       {d['display_name']:<35} {d['works_count']:>10,} papers")
    print("=" * 60)


In [10]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Fetch all four levels ──
raw_domains   = fetch_all("domains")
raw_fields    = fetch_all("fields")
raw_subfields = fetch_all("subfields")
raw_topics    = fetch_all("topics")

# ── Flatten ──
domains   = [flatten_item(x, "domains")   for x in raw_domains]
fields    = [flatten_item(x, "fields")    for x in raw_fields]
subfields = [flatten_item(x, "subfields") for x in raw_subfields]
topics    = [flatten_item(x, "topics")    for x in raw_topics]

# Filter topics by min works if configured
if MIN_WORKS_FOR_GALAXY > 0:
    topics = [t for t in topics if t["works_count"] >= MIN_WORKS_FOR_GALAXY]

# ── Save individual level files ──
for name, data in [("domains", domains), ("fields", fields),
                   ("subfields", subfields), ("topics", topics)]:
    save_json(data, os.path.join(OUTPUT_DIR, f"{name}.json"))
    save_csv(data, os.path.join(OUTPUT_DIR, f"{name}.csv"))

# ── Save nested hierarchy ──
hierarchy = build_nested_hierarchy(domains, fields, subfields, topics)
save_json(hierarchy, os.path.join(OUTPUT_DIR, "taxonomy_summary.json"))

# ── Save full flat CSV with complete ancestry ──
if SAVE_FULL_FLAT_CSV:
    flat_path = os.path.join(OUTPUT_DIR, "topics_full_ancestry.csv")
    columns = [
        "id", "display_name", "description", "works_count", "cited_by_count",
        "subfield_id", "subfield_name",
        "field_id", "field_name",
        "domain_id", "domain_name",
        "keywords", "wikipedia",
    ]
    with open(flat_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(sorted(topics, key=lambda x: -x["works_count"]))
    print(f"  Saved: {flat_path}")

print_summary(domains, fields, subfields, topics)
print(f"\n[done] All files written to: {os.path.abspath(OUTPUT_DIR)}/")


[domains] Fetching…
  Page 1: +4 items  (total so far: 4/4)
  Page 2: +0 items  (total so far: 4/4)
  -> Done: 4 domains fetched

[fields] Fetching…
  Page 1: +26 items  (total so far: 26/26)
  Page 2: +0 items  (total so far: 26/26)
  -> Done: 26 fields fetched

[subfields] Fetching…
  Page 1: +200 items  (total so far: 200/252)
  Page 2: +52 items  (total so far: 252/252)
  Page 3: +0 items  (total so far: 252/252)
  -> Done: 252 subfields fetched

[topics] Fetching…
  Page 1: +200 items  (total so far: 200/4516)
  Page 2: +200 items  (total so far: 400/4516)
  Page 3: +200 items  (total so far: 600/4516)
  Page 4: +200 items  (total so far: 800/4516)
  Page 5: +200 items  (total so far: 1000/4516)
  Page 6: +200 items  (total so far: 1200/4516)
  Page 7: +200 items  (total so far: 1400/4516)
  Page 8: +200 items  (total so far: 1600/4516)
  Page 9: +200 items  (total so far: 1800/4516)
  Page 10: +200 items  (total so far: 2000/4516)
  Page 11: +200 items  (total so far: 2200/4516)

In [16]:
df_topics = pd.read_csv("openalex_taxonomy/topics.csv")
df_subfields = pd.read_csv("openalex_taxonomy/subfields.csv")


In [18]:
df_topics

,id,display_name,description,works_count,cited_by_count,domain_id,domain_name,field_id,field_name,subfield_id,subfield_name,keywords,wikipedia
0,T14423,Military Technology and Strategies,This cluster of papers covers various aspects ...,22340475,817915,domains/3,Physical Sciences,fields/22,Engineering,subfields/2202,Aerospace Engineering,Air Force; Modernization; Warfare; Unmanned Ae...,NaN
1,T10346,Magnetic confinement fusion research,This cluster of papers covers a wide range of ...,9280538,2566013,domains/3,Physical Sciences,fields/31,Physics and Astronomy,subfields/3106,Nuclear and High Energy Physics,Turbulence; Tokamak; Transport; MHD Stability;...,NaN
2,T13370,Diverse Scientific and Economic Studies,This cluster of papers covers topics related t...,5031903,638513,domains/2,Social Sciences,fields/20,"Economics, Econometrics and Finance",subfields/2002,Economics and Econometrics,Financial Analysis; Monetary Policy; Asset Pri...,NaN
3,T12157,Geochemistry and Geologic Mapping,This cluster of papers focuses on the applicat...,3955869,1672923,domains/3,Physical Sciences,fields/17,Computer Science,subfields/1702,Artificial Intelligence,Machine Learning; Mineral Prospectivity; Remot...,NaN
4,T10451,Mycorrhizal Fungi and Plant Interactions,This cluster of papers explores the diverse in...,3216925,1983436,domains/1,Life Sciences,fields/11,Agricultural and Biological Sciences,subfields/1110,Plant Science,Mycorrhizal Fungi; Fungal Diversity; Plant Int...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4511,T14253,Latin American social science,This cluster of papers explores the dynamics o...,1351,1757,domains/2,Social Sciences,fields/33,Social Sciences,subfields/3312,Sociology and Political Science,Social Movements; Cultural Resistance; Citizen...,NaN
4512,T14027,21st Century Education and Governance,This cluster of papers covers a wide range of ...,1245,2401,domains/2,Social Sciences,fields/33,Social Sciences,subfields/3303,Development,Sustainable Development Goals; Environmental P...,NaN
4513,T13227,Diverse Global Economic and Educational Challe...,This cluster of papers explores the intersecti...,1221,3832,domains/2,Social Sciences,fields/20,"Economics, Econometrics and Finance",subfields/2002,Economics and Econometrics,Sustainable Development; Economic Growth; Educ...,NaN
4514,T13676,Educational and Technological Research,This cluster of papers explores the integratio...,1219,11992,domains/3,Physical Sciences,fields/17,Computer Science,subfields/1710,Information Systems,Big Data; Machine Learning; Education; Learnin...,NaN


In [14]:
df_topics.groupby("subfield_name")["works_count"].sum().sort_values(ascending=False)
df_topics[df_topics["domain_name"] == "Physical Sciences"].nlargest(20, "works_count")

,id,display_name,description,works_count,cited_by_count,domain_id,domain_name,field_id,field_name,subfield_id,subfield_name,keywords,wikipedia
0,T14423,Military Technology and Strategies,This cluster of papers covers various aspects ...,22340475,817915,domains/3,Physical Sciences,fields/22,Engineering,subfields/2202,Aerospace Engineering,Air Force; Modernization; Warfare; Unmanned Ae...,NaN
1,T10346,Magnetic confinement fusion research,This cluster of papers covers a wide range of ...,9280538,2566013,domains/3,Physical Sciences,fields/31,Physics and Astronomy,subfields/3106,Nuclear and High Energy Physics,Turbulence; Tokamak; Transport; MHD Stability;...,NaN
3,T12157,Geochemistry and Geologic Mapping,This cluster of papers focuses on the applicat...,3955869,1672923,domains/3,Physical Sciences,fields/17,Computer Science,subfields/1702,Artificial Intelligence,Machine Learning; Mineral Prospectivity; Remot...,NaN
6,T11367,Particle accelerators and beam dynamics,This cluster of papers focuses on advancements...,1474293,406003,domains/3,Physical Sciences,fields/22,Engineering,subfields/2202,Aerospace Engineering,Superconducting Cavities; Negative Ion Sources...,NaN
7,T10895,Species Distribution and Climate Change,This cluster of papers focuses on species dist...,1385387,2471023,domains/3,Physical Sciences,fields/23,Environmental Science,subfields/2302,Ecological Modeling,Species Distribution Modeling; Climate Change;...,NaN
9,T12692,Magnetic Field Sensors Techniques,This cluster of papers covers advances in magn...,1017886,154730,domains/3,Physical Sciences,fields/22,Engineering,subfields/2208,Electrical and Electronic Engineering,Magnetic Sensors; Fluxgate Sensors; Hall-effec...,NaN
11,T13638,Human auditory perception and evaluation,This cluster of papers explores the perception...,993677,264527,domains/3,Physical Sciences,fields/22,Engineering,subfields/2202,Aerospace Engineering,Hypersonic Effect; Ultrasonic Perception; Brai...,NaN
12,T10781,Plasma Diagnostics and Applications,This cluster of papers covers a wide range of ...,980533,894099,domains/3,Physical Sciences,fields/22,Engineering,subfields/2208,Electrical and Electronic Engineering,Plasma Etching; Semiconductor Industry; Atomic...,NaN
13,T12760,Laser Design and Applications,This cluster of papers focuses on the developm...,870949,439633,domains/3,Physical Sciences,fields/22,Engineering,subfields/2208,Electrical and Electronic Engineering,Electric Discharge; Laser Oscillation; Atomic ...,NaN
14,T11881,Crystallization and Solubility Studies,This cluster of papers focuses on the crystall...,778871,810226,domains/3,Physical Sciences,fields/25,Materials Science,subfields/2505,Materials Chemistry,Crystallization; Nucleation; Solubility; Polym...,NaN


In [17]:
topic_counts = df_topics.groupby("subfield_id").size().rename("topic_count")
df_subfields = df_subfields.join(topic_counts, on="id")
df_subfields.sort_values("works_count", ascending=False)

,id,display_name,description,works_count,cited_by_count,domain_id,domain_name,field_id,field_name,topic_count
0,subfields/2202,Aerospace Engineering,branch of engineering,27236109,17216179,domains/3,Physical Sciences,fields/22,Engineering,44
1,subfields/3312,Sociology and Political Science,academic disciplines concerned with society an...,14947779,65710629,domains/2,Social Sciences,fields/33,Social Sciences,224
2,subfields/3106,Nuclear and High Energy Physics,physics of elementary particles at high energies,12325910,22767715,domains/3,Physical Sciences,fields/31,Physics and Astronomy,13
3,subfields/1312,Molecular Biology,branch of biology that deals with the molecula...,10827318,184457808,domains/1,Life Sciences,fields/13,"Biochemistry, Genetics and Molecular Biology",135
4,subfields/2002,Economics and Econometrics,"social science that studies the production, di...",10722114,36998103,domains/2,Social Sciences,fields/20,"Economics, Econometrics and Finance",79
...,...,...,...,...,...,...,...,...,...,...
247,subfields/1108,Horticulture,agriculture of plants,44757,158255,domains/1,Life Sciences,fields/11,Agricultural and Biological Sciences,1
248,subfields/3102,Acoustics and Ultrasonics,science that deals with the study of all mecha...,26065,359484,domains/3,Physical Sciences,fields/31,Physics and Astronomy,1
249,subfields/2605,Computational Mathematics,area of mathematics,23218,282984,domains/3,Physical Sciences,fields/26,Mathematics,1
250,subfields/3002,Drug Discovery,the process by which new candidate medications...,13234,4160,domains/1,Life Sciences,fields/30,"Pharmacology, Toxicology and Pharmaceutics",1


In [20]:
pip install altair

   ---------------------------------------- 0.0/797.0 kB ? eta -:--:--
   ------- -------------------------------- 153.6/797.0 kB 4.6 MB/s eta 0:00:01
   --------------------------------------- 797.0/797.0 kB 12.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/449.4 kB ? eta -:--:--
   --------------------------------------- 449.4/449.4 kB 27.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

ps    = df_topics[df_topics['domain_name'] == 'Physical Sciences'].copy()
ps_sf = df_subfields[df_subfields['domain_name'] == 'Physical Sciences'].copy()

In [22]:
# Build one aggregated dataframe with all 3 levels
f_agg = ps.groupby('field_name', as_index=False)['works_count'].sum()
f_agg['level'] = 'Fields'
f_agg['label'] = f_agg['field_name']
f_agg['field'] = f_agg['field_name']

sf_agg = ps.groupby(['subfield_name', 'field_name'], as_index=False)['works_count'].sum()
sf_agg['level'] = 'Subfields'
sf_agg['label'] = sf_agg['subfield_name']
sf_agg['field'] = sf_agg['field_name']

t_agg = ps[['display_name', 'field_name', 'works_count']].copy()
t_agg['level'] = 'Topics'
t_agg['label'] = t_agg['display_name']
t_agg['field'] = t_agg['field_name']

hier = pd.concat([
    f_agg[['level', 'label', 'works_count', 'field']],
    sf_agg[['level', 'label', 'works_count', 'field']],
    t_agg[['level', 'label', 'works_count', 'field']],
], ignore_index=True)

alt.Chart(hier).mark_bar().encode(
    x=alt.X('level:N', sort=['Fields', 'Subfields', 'Topics'], title=None),
    y=alt.Y('sum(works_count):Q', title='Total Works', axis=alt.Axis(format='~s')),
    color=alt.Color('field:N', title='Field'),
    order=alt.Order('field:N'),
    tooltip=[
        alt.Tooltip('field:N', title='Field'),
        alt.Tooltip('level:N', title='Level'),
        alt.Tooltip('sum(works_count):Q', title='Works', format=','),
    ]
).properties(
    title='Physical Sciences — Works by Hierarchy Level (stacked by field)',
    width=350, height=450
)

alt.Chart(...)

In [23]:
sf_by_field = ps.groupby(['field_name', 'subfield_name'], as_index=False)['works_count'].sum()

alt.Chart(sf_by_field).mark_bar().encode(
    x=alt.X('works_count:Q', title='Total Works', axis=alt.Axis(format='~s')),
    y=alt.Y('field_name:N', sort='-x', title=None),
    color=alt.Color('subfield_name:N', title='Subfield',
                    legend=alt.Legend(columns=2, symbolLimit=80)),
    tooltip=[
        alt.Tooltip('field_name:N', title='Field'),
        alt.Tooltip('subfield_name:N', title='Subfield'),
        alt.Tooltip('works_count:Q', title='Works', format=','),
    ]
).properties(
    title='Physical Sciences — Works per Field, broken down by Subfield',
    width=600, height=300
)

alt.Chart(...)

In [24]:
alt.Chart(ps_sf).mark_circle(size=120, opacity=0.85).encode(
    x=alt.X('works_count:Q',
            scale=alt.Scale(type='log'),
            axis=alt.Axis(format='~s'),
            title='Works Count (log scale)'),
    y=alt.Y('display_name:N',
            sort=alt.EncodingSortField('works_count', order='descending'),
            title=None),
    color=alt.Color('field_name:N', title='Field'),
    tooltip=[
        alt.Tooltip('display_name:N', title='Subfield'),
        alt.Tooltip('field_name:N', title='Field'),
        alt.Tooltip('works_count:Q', title='Works', format=','),
        alt.Tooltip('cited_by_count:Q', title='Citations', format=','),
    ]
).properties(
    title='Physical Sciences Subfields — potential Mapo galaxies, sorted by size',
    width=550,
    height=len(ps_sf) * 14
).interactive()

alt.Chart(...)

In [29]:
top50 = ps.nlargest(50, 'works_count')

alt.Chart(top50).mark_circle(opacity=0.8).encode(
    x=alt.X('field_name:N', title=None, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('subfield_name:N', title=None,
            sort=alt.EncodingSortField('works_count', order='descending')),
    size=alt.Size('works_count:Q',
                  scale=alt.Scale(range=[200, 3000]),
                  title='Works'),
    color=alt.Color('field_name:N', legend=None),
    tooltip=[
        alt.Tooltip('display_name:N', title='Topic'),
        alt.Tooltip('subfield_name:N', title='Subfield'),
        alt.Tooltip('field_name:N', title='Field'),
        alt.Tooltip('works_count:Q', title='Works', format=','),
    ]
).properties(
    title='Top 50 Physical Sciences Topics — bubble size = works count',
    width=550, height=500
).interactive()

alt.Chart(...)

# Import a new topic

In [ ]:
# ==================== CONFIGURATION ====================
# Edit these variables before running the pipeline.

TOPIC = "Particle Physics"            # Topic name, e.g. "Astrophysics", "Nuclear Physics"
DB    = "papers_particle_physics.db"  # SQLite DB filename
EMAIL = "tom.hirsch3000@gmail.com"    # Email for OpenAlex polite pool
API_KEY = None                        # OpenAlex API key (None = use polite pool only)

# --- Import size control ---
TOP_N_PAPERS = 500    # Import only the top N most-cited papers (0 = no limit, import everything)
                      # When set, ignores year batching and does a single query sorted by citations

# --- Import batching (only used when TOP_N_PAPERS = 0) ---
YEAR_START       = 1800   # First publication year to import
YEAR_END         = 2026   # Last publication year to import
YEAR_BATCH_SIZE  = 1      # Years per import batch (use 10+ for sparse topics)
PAPERS_PER_BATCH = 0      # Max papers per year-range batch (0 = no limit)

# --- AI sampling ---
AI_SAMPLE     = 500   # Total papers to run AI on (0 = all)
AI_BATCH_SIZE = 200   # Papers per AI subprocess call (reduce if Ollama is slow)

# --- Visualization ---
MIN_CITATIONS = 0     # Min citations to include in visualization

# --- Step control ---
SKIP_STEPS  = []      # Steps to skip entirely, e.g. [3, 7]
FORCE_STEPS = []      # Steps to force re-run, e.g. [4]
ONLY_STEP   = None    # Run only this step number (None = run all)
RESET_IMPORT    = False
RESET_CITATIONS = False
SKIP_MISLABEL   = True

print(f"Configuration loaded: topic={TOPIC!r}, db={DB!r}")
if TOP_N_PAPERS > 0:
    print(f"Import mode: top {TOP_N_PAPERS} papers by citation count")
else:
    print(f"Import mode: all papers, year batches {YEAR_START}-{YEAR_END}")

In [ ]:
import datetime
import glob
import json
import sqlite3
import subprocess
import sys
import os
from pathlib import Path

# In a notebook, use the notebook's directory instead of __file__
SCRIPT_DIR = Path(os.getcwd())
DATA_DIR = SCRIPT_DIR / "data"
CHECKPOINT_VERSION = 2

# Derived values
slug = TOPIC.lower().replace(" ", "_")
DATA_DIR.mkdir(exist_ok=True)
nodes_out = f"data/{slug}_nodes.json"
edges_out = f"data/{slug}_edges.json"
meta_out  = f"data/{slug}_metadata.json"
FRONTEND_DIR = str(SCRIPT_DIR.parent / "arxiv-3d-frontend" / "public")

print(f"SCRIPT_DIR : {SCRIPT_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"Output     : {nodes_out}, {edges_out}, {meta_out}")
print(f"Frontend   : {FRONTEND_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions (checkpoint, DB queries, subprocess runner)
# ---------------------------------------------------------------------------

def checkpoint_path(db):
    return SCRIPT_DIR / f"{Path(db).stem}_checkpoint.json"

def load_checkpoint(db):
    p = checkpoint_path(db)
    if p.exists():
        try:
            with open(p) as f:
                data = json.load(f)
            if data.get("version") == CHECKPOINT_VERSION:
                return data
            print(f"[checkpoint] Version mismatch — starting fresh")
        except Exception as e:
            print(f"[checkpoint] Could not read {p}: {e} — starting fresh")
    return {"version": CHECKPOINT_VERSION, "import_batches_done": [], "steps_done": [], "ai_processed": 0}

def save_checkpoint(db, cp):
    cp["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    with open(checkpoint_path(db), "w") as f:
        json.dump(cp, f, indent=2)

def mark_step_done(db, cp, step):
    if step not in cp["steps_done"]:
        cp["steps_done"].append(step)
    save_checkpoint(db, cp)

def mark_import_batch_done(db, cp, batch_key_str):
    if batch_key_str not in cp["import_batches_done"]:
        cp["import_batches_done"].append(batch_key_str)
    save_checkpoint(db, cp)

def count_unprocessed_ai(db):
    db_path = SCRIPT_DIR / db
    if not db_path.exists():
        return 0
    try:
        conn = sqlite3.connect(str(db_path))
        cols = {r[1] for r in conn.execute("PRAGMA table_info(papers)").fetchall()}
        if "AI_field_list" not in cols:
            n = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
        else:
            n = conn.execute("""
                SELECT COUNT(*) FROM papers
                WHERE AI_field_list IS NULL OR AI_field_list = '[]'
                   OR AI_summary IS NULL OR TRIM(AI_summary) = ''
            """).fetchone()[0]
        conn.close()
        return n
    except Exception as e:
        print(f"[warn] Could not query DB for AI count: {e}")
        return 0

def count_total_papers(db):
    db_path = SCRIPT_DIR / db
    if not db_path.exists():
        return 0
    try:
        conn = sqlite3.connect(str(db_path))
        n = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
        conn.close()
        return n
    except Exception:
        return 0

def run_cmd(cmd, label, fatal=True):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  $ {' '.join(str(c) for c in cmd)}")
    print(f"{'='*60}")
    result = subprocess.run([str(c) for c in cmd], cwd=SCRIPT_DIR)
    if result.returncode != 0:
        print(f"\n[FAIL] Exit code {result.returncode}")
        if fatal:
            raise RuntimeError(f"Step failed: {label}. Fix the issue and re-run this cell.")
        return False
    print(f"[OK] Done.")
    return True

def make_year_batches(year_start, year_end, batch_size):
    batches = []
    y = year_start
    while y <= year_end:
        end = min(y + batch_size - 1, year_end)
        batches.append((y, end))
        y = end + 1
    return batches

def batch_key(from_year, to_year):
    return f"{from_year}-{to_year}"

def find_galaxy_args():
    node_files = sorted(glob.glob(str(DATA_DIR / "*_nodes.json")))
    galaxies = []
    for idx, nodes_path in enumerate(node_files, start=1):
        nodes_file = Path(nodes_path).name
        base = nodes_file.replace("_nodes.json", "")
        edges_file = f"{base}_edges.json"
        meta_file = f"{base}_metadata.json"
        name = base.replace("_", " ").title()
        if not (DATA_DIR / edges_file).exists() or not (DATA_DIR / meta_file).exists():
            continue
        galaxies.append(f"{idx}:{name}:data/{nodes_file}:data/{edges_file}:data/{meta_file}")
    return galaxies

def should_run(step, cp):
    if ONLY_STEP is not None:
        return step == ONLY_STEP
    if step in SKIP_STEPS:
        return False
    if step in FORCE_STEPS:
        return True
    if step in cp["steps_done"]:
        print(f"[checkpoint] Step {step} already done — skipping. Add {step} to FORCE_STEPS to re-run.")
        return False
    return True

# Load checkpoint
cp = load_checkpoint(DB)
cp.setdefault("topic", TOPIC)
cp.setdefault("db", DB)
save_checkpoint(DB, cp)

print(f"Checkpoint : {checkpoint_path(DB).name}")
print(f"Steps done : {cp['steps_done']}")
print(f"Import batches done: {len(cp['import_batches_done'])}")
print(f"AI papers processed: {cp['ai_processed']}")
print(f"Total papers in DB : {count_total_papers(DB)}")

## Step 1 — Import papers from OpenAlex
- **TOP_N_PAPERS > 0**: imports the top N most-cited papers in a single query (no year batching)
- **TOP_N_PAPERS = 0**: downloads all papers in year-range batches, checkpointed per batch

In [ ]:
# STEP 1 — Import papers
if should_run(1, cp):
    if TOP_N_PAPERS > 0:
        # ---------- Top-N mode: single query, sorted by cited_by_count:desc ----------
        print(f"[step 1] Importing top {TOP_N_PAPERS} most-cited papers for '{TOPIC}'...")
        cmd = [
            sys.executable, "import_openalex.py",
            "--topic-name", TOPIC,
            "--db", DB,
            "--email", EMAIL,
            "--sample", str(TOP_N_PAPERS),
        ]
        if API_KEY:
            cmd += ["--api-key", API_KEY]
        if RESET_IMPORT:
            cmd.append("--reset")
        run_cmd(cmd, f"1 — Import top {TOP_N_PAPERS} by citations")
        mark_step_done(DB, cp, 1)
        print(f"\n[step 1] Import complete. Total papers: {count_total_papers(DB)}")
    else:
        # ---------- Year-batch mode ----------
        batches = make_year_batches(YEAR_START, YEAR_END, YEAR_BATCH_SIZE)
        done_keys = set(cp["import_batches_done"])

        if 1 in FORCE_STEPS:
            print("[info] FORCE_STEPS includes 1: clearing import batch history")
            cp["import_batches_done"] = []
            done_keys = set()
            save_checkpoint(DB, cp)

        pending = [(f, t) for (f, t) in batches if batch_key(f, t) not in done_keys]
        print(f"[step 1] {len(batches)} year-range batches total, "
              f"{len(done_keys)} already done, {len(pending)} to go.")

        reset_import = RESET_IMPORT
        for from_y, to_y in pending:
            bk = batch_key(from_y, to_y)
            cmd = [
                sys.executable, "import_openalex.py",
                "--topic-name", TOPIC,
                "--db", DB,
                "--email", EMAIL,
                "--from-year", str(from_y),
                "--to-year",   str(to_y),
            ]
            if API_KEY:
                cmd += ["--api-key", API_KEY]
            if PAPERS_PER_BATCH > 0:
                cmd += ["--sample", str(PAPERS_PER_BATCH)]
            if reset_import:
                cmd.append("--reset")
                reset_import = False

            ok = run_cmd(cmd, f"1 — Import {from_y}–{to_y}", fatal=False)
            if ok:
                mark_import_batch_done(DB, cp, bk)
                print(f"[checkpoint] Batch {bk} done. Total papers: {count_total_papers(DB)}")
            else:
                print(f"[warn] Batch {bk} failed. Re-run this cell to resume.")
                break

        mark_step_done(DB, cp, 1)
        print(f"\n[step 1] Import complete. Total papers: {count_total_papers(DB)}")
else:
    print(f"Step 1 skipped. Papers in DB: {count_total_papers(DB)}")

## Step 2 — Build citation edges

In [ ]:
# STEP 2 — Build citation edges
if should_run(2, cp):
    cmd = [sys.executable, "rebuild_citations_openalex.py", "--db", DB]
    if RESET_CITATIONS:
        cmd.append("--reset")
    run_cmd(cmd, "2 — Rebuild citation edges")
    mark_step_done(DB, cp, 2)
else:
    print("Step 2 skipped.")

## Step 3 — Fetch missing abstracts (Semantic Scholar / arXiv)

In [ ]:
# STEP 3 — Fetch missing abstracts
if should_run(3, cp):
    run_cmd(
        [sys.executable, "fetch_abstracts_s2_arxiv.py", "--db", DB],
        "3 — Fetch missing abstracts (Semantic Scholar / arXiv)"
    )
    mark_step_done(DB, cp, 3)
else:
    print("Step 3 skipped.")

## Step 4 — AI metadata (sampled, batched)
Runs AI classification on papers. Batched with checkpoint so you can stop/resume.

In [ ]:
# STEP 4 — AI metadata (sampled, batched loop)
if should_run(4, cp):
    target = AI_SAMPLE
    batch = AI_BATCH_SIZE

    if 4 in FORCE_STEPS:
        cp["ai_processed"] = 0
        save_checkpoint(DB, cp)

    already_done = cp["ai_processed"]
    remaining_target = (target - already_done) if target > 0 else float("inf")
    unprocessed = count_unprocessed_ai(DB)

    print(f"[step 4] AI metadata:")
    print(f"         Unprocessed papers in DB : {unprocessed}")
    print(f"         Already processed (this run): {already_done}")
    print(f"         Target (AI_SAMPLE)       : {target if target > 0 else 'unlimited'}")
    print(f"         Batch size               : {batch}")

    if remaining_target <= 0:
        print(f"[checkpoint] Already reached AI sample target ({target}). "
              f"Add 4 to FORCE_STEPS or increase AI_SAMPLE.")
    elif unprocessed == 0:
        print("[info] No unprocessed papers found — skipping AI step.")
        mark_step_done(DB, cp, 4)
    else:
        processed_this_run = 0
        while True:
            to_process = batch if target == 0 else min(batch, int(remaining_target) - processed_this_run)
            if to_process <= 0:
                break
            unprocessed = count_unprocessed_ai(DB)
            if unprocessed == 0:
                print("[info] All papers now have AI metadata.")
                break

            print(f"\n[step 4] Running AI on next {to_process} papers "
                  f"({processed_this_run + already_done} done so far, "
                  f"{unprocessed} remaining in DB)...")

            run_cmd(
                [sys.executable, "process_ai_metadata.py",
                 "--db", DB,
                 "--only-unprocessed", "1",
                 "--limit", str(to_process)],
                f"4 — AI metadata (batch of {to_process})"
            )

            processed_this_run += to_process
            cp["ai_processed"] = already_done + processed_this_run
            save_checkpoint(DB, cp)

            if target > 0 and processed_this_run >= int(remaining_target):
                print(f"[info] Reached AI sample target ({target} total).")
                break

        mark_step_done(DB, cp, 4)
        print(f"\n[step 4] AI done. Total processed: {cp['ai_processed']}")
else:
    print("Step 4 skipped.")

## Step 5 — Clean and standardize field classifications

In [ ]:
# STEP 5 — Clean and standardize fields
if should_run(5, cp):
    cmd = [
        sys.executable, "clean_and_categorize.py",
        "--db", DB,
        "--field", TOPIC,
    ]
    if SKIP_MISLABEL:
        cmd.append("--skip-mislabel")
    if AI_SAMPLE > 0:
        cmd += ["--limit", str(AI_SAMPLE)]
    run_cmd(cmd, "5 — Clean and standardize field classifications")
    mark_step_done(DB, cp, 5)
else:
    print("Step 5 skipped.")

## Step 6 — Build frontend JSON (nodes, edges, metadata)

In [ ]:
# STEP 6 — Build frontend JSON
if should_run(6, cp):
    cmd = [
        sys.executable, "build_frontend_json.py",
        "--db", DB,
        "--output-nodes", nodes_out,
        "--output-edges", edges_out,
        "--output-metadata", meta_out,
        "--min-citations", str(MIN_CITATIONS),
        "--compute-clusters",
    ]
    if AI_SAMPLE > 0:
        cmd += ["--top-n", str(AI_SAMPLE)]
    if FRONTEND_DIR:
        cmd += ["--frontend-dir", FRONTEND_DIR]
    run_cmd(cmd, "6 — Build nodes/edges JSON for frontend")
    mark_step_done(DB, cp, 6)
else:
    print("Step 6 skipped.")

## Step 7 — Rebuild universe view
Discovers all topic JSON files and builds the combined universe.json.

In [ ]:
# STEP 7 — Rebuild universe view
if should_run(7, cp):
    galaxies = find_galaxy_args()
    if not galaxies:
        print("[warn] No topic JSON files found — skipping universe build.")
    else:
        cmd = [
            sys.executable, "build_universe_json.py",
            "--email", EMAIL,
            "--output", "data/universe.json",
        ]
        if FRONTEND_DIR:
            cmd += ["--frontend-dir", FRONTEND_DIR]
        for g in galaxies:
            cmd += ["--galaxies", g]
        run_cmd(cmd, "7 — Build universe view")
    mark_step_done(DB, cp, 7)
else:
    print("Step 7 skipped.")

## Summary

In [ ]:
# Reload checkpoint and show final status
cp = load_checkpoint(DB)
print(f"{'='*60}")
print(f"  Pipeline status: {TOPIC}")
print(f"  Papers in DB     : {count_total_papers(DB)}")
print(f"  AI processed     : {cp['ai_processed']}")
print(f"  Steps done       : {sorted(cp['steps_done'])}")
print(f"{'='*60}")